# Phase 2 — 캘리브레이션과 좌표계 (실무 방식 재구성)

정답지 `notebooks/02_calibration_and_frames.ipynb` 와 `src/perceptrack3d/geometry/{calibration.py, transforms.py}` 가 하는 일을
**검색해서 원본을 가져다 붙이는 방식**으로 다시 만든다. 조각마다 A(원본 발췌) → B(우리 데이터에 적용) → C(정답지와 검증).

목표 출력: `T_velo_to_cam (4×4)`, `R_rect_00 (4×4)`, `P_rect_02 (3×4)`, 그리고 세 개를 곱한 `P_velo_to_img (3×4)`.

## 이 노트북에서 가져오는 것

| 조각 | 출처 | 라이선스 | 왜 |
|---|---|---|---|
| 1. 캘리브레이션 텍스트 파싱 | `readme.txt` "Sensor Calibration" 절 + pykitti `utils.read_calib_file` | KITTI(연구용) / MIT | `key: 숫자들` 형식과 각 키의 뜻(R, T, R_rect_xx, P_rect_xx)을 문서로 확인 |
| 2. `T_velo_to_cam` 4×4 조립 | devkit `loadCalibrationRigid.m` + pykitti `utils.transform_from_rot_trans` + Wikipedia "Affine transformation" | KITTI / MIT / CC BY-SA | 회전 R 과 이동 T 를 동차 행렬 하나로 |
| 3. `R_rect_00` 4×4 확장, `P_rect_02` | pykitti `raw._load_calib_rigid`, `raw._load_calib_cam_to_cam` + `readme.txt` 230~231행 | MIT / KITTI | 3×3 정류 회전을 4×4 로 채우는 관례와 pykitti 의 이름 규칙(`P_rect_20` = 파일의 `P_rect_02`) |
| 4. `P_velo_to_img` 조립 | `readme.txt` "example transformations" 절 + devkit `run_demoVelodyne.m` | KITTI | 투영식 `P_rect_xx * R_rect_00 * (R|T)_velo_to_cam` 의 순서와 "항상 `R_rect_00`" 규칙 |
| 5. 역변환과 직교 보정 | Wikipedia "Orthogonal matrix", "Orthogonal Procrustes problem" + pykitti 의 `np.linalg.inv` 사용부 | CC BY-SA 4.0 / MIT | rect → velo 로 되돌릴 때 필요. 텍스트 반올림 때문에 Rᵀ 역이 정확하지 않은 함정 |
| 6. 부호 확인 실험 | `readme.txt` "Coordinate Systems" 절 | KITTI | 붙인 행렬이 맞는지는 아는 점 몇 개로 바로 확인 |

용어
- **강체 변환(rigid transform)**: 회전 + 이동. $p' = Rp + t$. 길이·각도 보존.
- **동차 좌표(homogeneous)**: 점 뒤에 1 을 붙인 $[x, y, z, 1]^T$. 회전과 이동을 4×4 행렬 곱 하나로 쓴다.
- **정류(rectification)**: 스테레오 두 카메라의 이미지 평면이 평행하도록 가상으로 회전시키는 것. KITTI `image_0x/data` 는 이미 정류된 이미지.

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from perceptrack3d.config import load_config   # 경로는 여기서만
cfg = load_config()
CALIB_DIR = Path(cfg["dataset"]["calib_dir"])
np.set_printoptions(precision=6, suppress=True, linewidth=120)
print("calib_dir:", CALIB_DIR)
print(sorted(p.name for p in CALIB_DIR.glob("*.txt")))

calib_dir: /home/ingon/datasets/KITTI/raw/2011_09_26
['calib_cam_to_cam.txt', 'calib_imu_to_velo.txt', 'calib_velo_to_cam.txt']


### 조각 1. 캘리브레이션 텍스트 파일을 dict 로 읽기

- **검색어**: `kitti calib_cam_to_cam.txt parse python`
- **출처**:
  - `readme.txt` "Sensor Calibration" 절 (162~204행) — https://s3.eu-central-1.amazonaws.com/avg-kitti/devkit_raw_data.zip — KITTI(연구용) — 2026-09-08 — `notebooks_2nd/sources/kitti_devkit/devkit/readme.txt`
  - pykitti `utils.py :: read_calib_file` — https://github.com/utiasSTARS/pykitti — MIT — 2026-09-08 — `notebooks_2nd/sources/pykitti/utils.py` 68~85행
- **왜 이걸 가져오나**: 파일은 `key: v1 v2 ...` 텍스트다. 행렬 shape 은 파일에 없고 readme 에만 있다 (`R_rect_xx: 3x3`, `P_rect_xx: 3x4`).
- **읽을 때 볼 곳**: readme 165~167행 "storing matrices in row-aligned order" → `reshape` 는 행 우선(기본값)으로 하면 된다. 187~194행: `R|T takes a point in Velodyne coordinates and transforms it into the coordinate system of the left video camera`. `read_calib_file` 의 `except ValueError: pass` — `calib_time: 15-Mar-2012 ...` 처럼 숫자가 아닌 줄을 건너뛴다.

readme.txt 원문 (165~194행 발췌):

```text
The sensor calibration zip archive contains files, storing matrices in
row-aligned order, meaning that the first values correspond to the first
row:

calib_cam_to_cam.txt: Camera-to-camera calibration
--------------------------------------------------

  - S_xx: 1x2 size of image xx before rectification
  - K_xx: 3x3 calibration matrix of camera xx before rectification
  - D_xx: 1x5 distortion vector of camera xx before rectification
  - R_xx: 3x3 rotation matrix of camera xx (extrinsic)
  - T_xx: 3x1 translation vector of camera xx (extrinsic)
  - S_rect_xx: 1x2 size of image xx after rectification
  - R_rect_xx: 3x3 rectifying rotation to make image planes co-planar
  - P_rect_xx: 3x4 projection matrix after rectification

Note: When using this dataset you will most likely need to access only
P_rect_xx, as this matrix is valid for the rectified image sequences.

calib_velo_to_cam.txt: Velodyne-to-camera registration
------------------------------------------------------

  - R: 3x3 rotation matrix
  - T: 3x1 translation vector
  - delta_f: deprecated
  - delta_c: deprecated

R|T takes a point in Velodyne coordinates and transforms it into the
coordinate system of the left video camera. Likewise it serves as a
representation of the Velodyne coordinate frame in camera coordinates.
```

In [2]:
# ORIGINAL: pykitti/utils.py :: read_calib_file (68~85행), MIT
def read_calib_file(filepath):
    """Read in a calibration file and parse into a dictionary."""
    data = {}

    with open(filepath, 'r') as f:
        for line in f.readlines():
            try:
                key, value = line.split(':', 1)
            except ValueError:
                key, value = line.split(' ', 1)
            # The only non-float values in these files are dates, which
            # we don't care about anyway
            try:
                data[key] = np.array([float(x) for x in value.split()])
            except ValueError:
                pass

    return data

In [3]:
print("=== calib_velo_to_cam.txt 원문 ===")
print((CALIB_DIR / "calib_velo_to_cam.txt").read_text())

vc = read_calib_file(CALIB_DIR / "calib_velo_to_cam.txt")
cc = read_calib_file(CALIB_DIR / "calib_cam_to_cam.txt")
print("velo_to_cam keys:", {k: v.shape for k, v in vc.items()}, " ('calib_time' 은 float 변환 실패로 빠짐)")
print("cam_to_cam keys :", len(cc), "개 —", [k for k in cc if k.endswith("_02") or k == "R_rect_00"])

# readme 의 shape 대로 되돌린다 (row-aligned → 기본 reshape)
print("\nR (3x3) =\n", vc["R"].reshape(3, 3))
print("T (3,)  =", vc["T"])
print("R_rect_00 (3x3) =\n", cc["R_rect_00"].reshape(3, 3))
print("P_rect_02 (3x4) =\n", cc["P_rect_02"].reshape(3, 4))
print("S_rect_02 (width, height) =", cc["S_rect_02"], " ← numpy 이미지 shape (H, W) 와 순서가 반대")

=== calib_velo_to_cam.txt 원문 ===
calib_time: 15-Mar-2012 11:37:16
R: 7.533745e-03 -9.999714e-01 -6.166020e-04 1.480249e-02 7.280733e-04 -9.998902e-01 9.998621e-01 7.523790e-03 1.480755e-02
T: -4.069766e-03 -7.631618e-02 -2.717806e-01
delta_f: 0.000000e+00 0.000000e+00
delta_c: 0.000000e+00 0.000000e+00

velo_to_cam keys: {'R': (9,), 'T': (3,), 'delta_f': (2,), 'delta_c': (2,)}  ('calib_time' 은 float 변환 실패로 빠짐)
cam_to_cam keys : 33 개 — ['R_rect_00', 'S_02', 'K_02', 'D_02', 'R_02', 'T_02', 'S_rect_02', 'R_rect_02', 'P_rect_02']

R (3x3) =
 [[ 0.007534 -0.999971 -0.000617]
 [ 0.014802  0.000728 -0.99989 ]
 [ 0.999862  0.007524  0.014808]]
T (3,)  = [-0.00407  -0.076316 -0.271781]
R_rect_00 (3x3) =
 [[ 0.999924  0.009838 -0.007445]
 [-0.00987   0.999942 -0.004278]
 [ 0.007403  0.004352  0.999963]]
P_rect_02 (3x4) =
 [[721.5377     0.       609.5593    44.85728 ]
 [  0.       721.5377   172.854      0.216379]
 [  0.         0.         1.         0.002746]]
S_rect_02 (width, height) = [1242.

In [4]:
# 검증: 정답지 read_calib_file 과 비교 (키 집합, 값)
from perceptrack3d.geometry.calibration import read_calib_file as ref_read_calib_file

for name in ("calib_velo_to_cam.txt", "calib_cam_to_cam.txt"):
    mine, ref = read_calib_file(CALIB_DIR / name), ref_read_calib_file(CALIB_DIR / name)
    same_keys = set(mine) == set(ref)
    same_vals = all(np.array_equal(mine[k], ref[k]) for k in mine)
    print(f"{name:22s} keys 동일 ? {same_keys} | 값 array_equal ? {same_vals} | dtype 정답지 {ref['R' if 'velo' in name else 'R_rect_00'].dtype}")
# 차이: 정답지는 'key: value' 의 ':' 만 구분자로 쓰고(line.partition), 없는 파일이면 FileNotFoundError 를 던진다. 값은 둘 다 float64.

calib_velo_to_cam.txt  keys 동일 ? True | 값 array_equal ? True | dtype 정답지 float64
calib_cam_to_cam.txt   keys 동일 ? True | 값 array_equal ? True | dtype 정답지 float64


### 조각 2. `R`, `T` → 4×4 동차 행렬 `T_velo_to_cam`

- **검색어**: `kitti velo_to_cam R T 4x4 homogeneous transformation matrix`
- **출처**:
  - devkit `matlab/loadCalibrationRigid.m` — KITTI(연구용) — 2026-09-08 — `notebooks_2nd/sources/kitti_devkit/devkit/matlab/loadCalibrationRigid.m` 10~13행
  - pykitti `utils.py :: transform_from_rot_trans` — https://github.com/utiasSTARS/pykitti — MIT — 2026-09-08 — `notebooks_2nd/sources/pykitti/utils.py` 61~65행
  - Wikipedia "Affine transformation — Augmented matrix" — https://en.wikipedia.org/wiki/Affine_transformation — CC BY-SA 4.0 — 2026-09-08 — `notebooks_2nd/sources/wikipedia/affine_transformation.txt`
- **왜 이걸 가져오나**: $p_{cam} = R p_{velo} + T$ 를 행렬 곱 하나로 쓰려면 `[[R, T], [0 0 0 1]]` 로 묶어야 한다. 이후 정류 회전·투영과 연쇄 곱을 하기 위한 준비.
- **읽을 때 볼 곳**: MATLAB 은 `Tr = [R T;0 0 0 1]` 한 줄. pykitti 는 `reshape(3, 3)`, `reshape(3, 1)` 뒤 `vstack/hstack`. Wikipedia: 점 뒤에 1 을 붙이면 `y = A x + b` 가 `[y;1] = [[A, b],[0,1]] [x;1]` 과 같다.

loadCalibrationRigid.m 원문 (10~13행):

```matlab
% read calibration
R  = readVariable(fid,'R',3,3);
T  = readVariable(fid,'T',3,1);
Tr = [R T;0 0 0 1];
```

pykitti 의 `transform_from_rot_trans` 가 정확히 같은 일을 numpy 로 한다 (`vstack((hstack([R, t]), [0, 0, 0, 1]))` = `[R T; 0 0 0 1]`).

In [5]:
# ORIGINAL: pykitti/utils.py :: transform_from_rot_trans (61~65행), MIT   — MATLAB `Tr = [R T;0 0 0 1]` 의 numpy 판
def transform_from_rot_trans(R, t):
    """Transforation matrix from rotation matrix and translation vector."""
    R = R.reshape(3, 3)
    t = t.reshape(3, 1)
    return np.vstack((np.hstack([R, t]), [0, 0, 0, 1]))

In [6]:
T_velo_to_cam = transform_from_rot_trans(vc['R'], vc['T'])      # 파일의 1차원 배열을 그대로 넣어도 안에서 reshape 한다
print("T_velo_to_cam (4x4) =\n", T_velo_to_cam)
print("shape:", T_velo_to_cam.shape, "| 마지막 행 [0 0 0 1] ?", np.array_equal(T_velo_to_cam[3], [0, 0, 0, 1]))

# 뜻 읽기: 4번째 열 = Velodyne 원점을 cam0 좌표로 옮긴 위치 (cam0: x 우, y 아래, z 전방)
t = T_velo_to_cam[:3, 3]
print(f"\nVelodyne 원점 (cam0 프레임) = {t.round(3)} m  →  카메라보다 {-t[2]:.2f} m 뒤(−z), {-t[1]*100:.1f} cm 위(−y)")

# 회전 부분이 "정확히" 직교인지 (조각 5 의 복선): 파일은 유효숫자 7자리로 반올림돼 있다
R = T_velo_to_cam[:3, :3]
print("|R Rᵀ − I|max =", f"{np.abs(R @ R.T - np.eye(3)).max():.1e}", "| det R =", f"{np.linalg.det(R):.9f}")

T_velo_to_cam (4x4) =
 [[ 0.007534 -0.999971 -0.000617 -0.00407 ]
 [ 0.014802  0.000728 -0.99989  -0.076316]
 [ 0.999862  0.007524  0.014808 -0.271781]
 [ 0.        0.        0.        1.      ]]
shape: (4, 4) | 마지막 행 [0 0 0 1] ? True

Velodyne 원점 (cam0 프레임) = [-0.004 -0.076 -0.272] m  →  카메라보다 0.27 m 뒤(−z), 7.6 cm 위(−y)
|R Rᵀ − I|max = 9.0e-08 | det R = 1.000000042


In [7]:
# 검증: 정답지 KittiCalibration.T_velo_to_cam 과 비교
from perceptrack3d.geometry.calibration import KittiCalibration

calib = KittiCalibration.from_dir(CALIB_DIR)
diff = np.abs(T_velo_to_cam - calib.T_velo_to_cam).max()
print("np.allclose(atol=1e-6) ?", np.allclose(T_velo_to_cam, calib.T_velo_to_cam, atol=1e-6), f"| max |차이| = {diff:.1e}")
print("정확히 같지는 않다: 정답지는 R 을 SVD 로 직교 보정(nearest_rotation)한다 → 차이 ~1e-8. 이유와 효과는 조각 5 에서 실험.")

np.allclose(atol=1e-6) ? True | max |차이| = 4.5e-08
정확히 같지는 않다: 정답지는 R 을 SVD 로 직교 보정(nearest_rotation)한다 → 차이 ~1e-8. 이유와 효과는 조각 5 에서 실험.


### 조각 3. `R_rect_00` 을 4×4 로 확장하고 `P_rect_02` 꺼내기 — pykitti `_load_calib_cam_to_cam`

- **검색어**: `pykitti calib R_rect_00 P_rect_20 T_cam2_velo`
- **출처**:
  - pykitti `raw.py :: raw._load_calib_rigid`, `raw._load_calib_cam_to_cam` — https://github.com/utiasSTARS/pykitti — MIT — 2026-09-08 — `notebooks_2nd/sources/pykitti/raw.py` 144~223행
  - `readme.txt` 230~231행: `Note that the (4x4) matrices above are padded with zeros and: R_rect_00(4,4) = (R|T)_velo_to_cam(4,4) = (R|T)_imu_to_velo(4,4) = 1.`
- **왜 이걸 가져오나**: `R_rect_00` 은 3×3 인데 4×4 `T_velo_to_cam` 과 곱하려면 4×4 로 채워야 한다. 어떻게 채우는지(`np.eye(4)` 에 넣기)와, 파일 키 이름과 라이브러리 변수 이름이 **다르다**는 것을 확인한다.
- **읽을 때 볼 곳**:
  - 174~176행 `R_rect_00 = np.eye(4); R_rect_00[0:3, 0:3] = np.reshape(filedata['R_rect_00'], (3, 3))` — readme 230~231행 그대로.
  - 166행 `P_rect_20 = np.reshape(filedata['P_rect_02'], (3, 4))` — **파일 키는 `P_rect_02`, pykitti 변수는 `P_rect_20`**. pykitti 는 `X_a_b` = "b 에서 a 로" 규칙(`T_cam0_velo` = velo → cam0)이라 카메라 번호를 앞에 둔다. 정답지는 `T_velo_to_cam` 처럼 "from → to" 를 이름에 쓴다. 남의 코드를 붙일 때 가장 자주 터지는 곳이 이 **이름의 방향**이다.
  - 189~203행: pykitti 는 `P_rect_20[0,3]/P_rect_20[0,0]` (= 기준선 baseline) 을 `T2` 로 빼내 `T_cam2_velo` 를 만든다. 조각 4 에서 readme 식과 비교한다.

In [8]:
# ORIGINAL: pykitti/raw.py :: raw._load_calib_rigid (144~148행), raw._load_calib_cam_to_cam (150~223행), MIT
def _load_calib_rigid(self, filename):
    """Read a rigid transform calibration file as a numpy.array."""
    filepath = os.path.join(self.calib_path, filename)
    data = utils.read_calib_file(filepath)
    return utils.transform_from_rot_trans(data['R'], data['T'])

def _load_calib_cam_to_cam(self, velo_to_cam_file, cam_to_cam_file):
    # We'll return the camera calibration as a dictionary
    data = {}

    # Load the rigid transformation from velodyne coordinates
    # to unrectified cam0 coordinates
    T_cam0unrect_velo = self._load_calib_rigid(velo_to_cam_file)
    data['T_cam0_velo_unrect'] = T_cam0unrect_velo

    # Load and parse the cam-to-cam calibration data
    cam_to_cam_filepath = os.path.join(self.calib_path, cam_to_cam_file)
    filedata = utils.read_calib_file(cam_to_cam_filepath)

    # Create 3x4 projection matrices
    P_rect_00 = np.reshape(filedata['P_rect_00'], (3, 4))
    P_rect_10 = np.reshape(filedata['P_rect_01'], (3, 4))
    P_rect_20 = np.reshape(filedata['P_rect_02'], (3, 4))
    P_rect_30 = np.reshape(filedata['P_rect_03'], (3, 4))

    data['P_rect_00'] = P_rect_00
    data['P_rect_10'] = P_rect_10
    data['P_rect_20'] = P_rect_20
    data['P_rect_30'] = P_rect_30

    # Create 4x4 matrices from the rectifying rotation matrices
    R_rect_00 = np.eye(4)
    R_rect_00[0:3, 0:3] = np.reshape(filedata['R_rect_00'], (3, 3))
    R_rect_10 = np.eye(4)
    R_rect_10[0:3, 0:3] = np.reshape(filedata['R_rect_01'], (3, 3))
    R_rect_20 = np.eye(4)
    R_rect_20[0:3, 0:3] = np.reshape(filedata['R_rect_02'], (3, 3))
    R_rect_30 = np.eye(4)
    R_rect_30[0:3, 0:3] = np.reshape(filedata['R_rect_03'], (3, 3))

    data['R_rect_00'] = R_rect_00
    data['R_rect_10'] = R_rect_10
    data['R_rect_20'] = R_rect_20
    data['R_rect_30'] = R_rect_30

    # Compute the rectified extrinsics from cam0 to camN
    T0 = np.eye(4)
    T0[0, 3] = P_rect_00[0, 3] / P_rect_00[0, 0]
    T1 = np.eye(4)
    T1[0, 3] = P_rect_10[0, 3] / P_rect_10[0, 0]
    T2 = np.eye(4)
    T2[0, 3] = P_rect_20[0, 3] / P_rect_20[0, 0]
    T3 = np.eye(4)
    T3[0, 3] = P_rect_30[0, 3] / P_rect_30[0, 0]

    # Compute the velodyne to rectified camera coordinate transforms
    data['T_cam0_velo'] = T0.dot(R_rect_00.dot(T_cam0unrect_velo))
    data['T_cam1_velo'] = T1.dot(R_rect_00.dot(T_cam0unrect_velo))
    data['T_cam2_velo'] = T2.dot(R_rect_00.dot(T_cam0unrect_velo))
    data['T_cam3_velo'] = T3.dot(R_rect_00.dot(T_cam0unrect_velo))

    # Compute the camera intrinsics
    data['K_cam0'] = P_rect_00[0:3, 0:3]
    data['K_cam1'] = P_rect_10[0:3, 0:3]
    data['K_cam2'] = P_rect_20[0:3, 0:3]
    data['K_cam3'] = P_rect_30[0:3, 0:3]

    # Compute the stereo baselines in meters by projecting the origin of
    # each camera frame into the velodyne frame and computing the distances
    # between them
    p_cam = np.array([0, 0, 0, 1])
    p_velo0 = np.linalg.inv(data['T_cam0_velo']).dot(p_cam)
    p_velo1 = np.linalg.inv(data['T_cam1_velo']).dot(p_cam)
    p_velo2 = np.linalg.inv(data['T_cam2_velo']).dot(p_cam)
    p_velo3 = np.linalg.inv(data['T_cam3_velo']).dot(p_cam)

    data['b_gray'] = np.linalg.norm(p_velo1 - p_velo0)  # gray baseline
    data['b_rgb'] = np.linalg.norm(p_velo3 - p_velo2)   # rgb baseline

    return data

In [9]:
import os                                              # CHANGED: 원본 파일 머리의 import
from types import SimpleNamespace

# 원본은 `import pykitti.utils as utils` 로 조각 1·2 의 함수를 부른다 → 우리가 붙인 함수를 같은 이름의 네임스페이스로 묶는다
utils = SimpleNamespace(read_calib_file=read_calib_file, transform_from_rot_trans=transform_from_rot_trans)   # CHANGED
# raw 객체 흉내: 메서드가 쓰는 속성 calib_path 와, 안에서 부르는 self._load_calib_rigid(...) 를 바인딩
self = SimpleNamespace(calib_path=str(CALIB_DIR))                                                                  # CHANGED
self._load_calib_rigid = lambda filename: _load_calib_rigid(self, filename)                                       # CHANGED

data = _load_calib_cam_to_cam(self, 'calib_velo_to_cam.txt', 'calib_cam_to_cam.txt')
print("pykitti 가 만드는 키:", sorted(data))

R_rect_00 = data['R_rect_00']                          # (4,4) — 3x3 회전을 eye(4) 에 넣은 것
P_rect_02 = data['P_rect_20']                          # (3,4) — 이름 주의: pykitti 'P_rect_20' == 파일 'P_rect_02'
print("\nR_rect_00 (4x4) =\n", R_rect_00)
print("P_rect_02 (3x4) =\n", P_rect_02)
f, cx, cy, tx = P_rect_02[0, 0], P_rect_02[0, 2], P_rect_02[1, 2], P_rect_02[0, 3]
print(f"\nf = {f:.2f} px, (cx, cy) = ({cx:.2f}, {cy:.2f}) px, tx/f = {tx / f:+.4f} m (cam2 가 cam0 에서 옆으로 떨어진 기준선)")
print("pykitti b_rgb (cam2–cam3 기준선) =", round(float(data['b_rgb']), 4), "m  (KITTI 공식 약 0.54 m)")
print("\n'T_cam0_velo_unrect' 가 우리 T_velo_to_cam 과 같은가 ?", np.array_equal(data['T_cam0_velo_unrect'], T_velo_to_cam), " ← 이름만 다르다 (cam0 ← velo)")

pykitti 가 만드는 키: ['K_cam0', 'K_cam1', 'K_cam2', 'K_cam3', 'P_rect_00', 'P_rect_10', 'P_rect_20', 'P_rect_30', 'R_rect_00', 'R_rect_10', 'R_rect_20', 'R_rect_30', 'T_cam0_velo', 'T_cam0_velo_unrect', 'T_cam1_velo', 'T_cam2_velo', 'T_cam3_velo', 'b_gray', 'b_rgb']

R_rect_00 (4x4) =
 [[ 0.999924  0.009838 -0.007445  0.      ]
 [-0.00987   0.999942 -0.004278  0.      ]
 [ 0.007403  0.004352  0.999963  0.      ]
 [ 0.        0.        0.        1.      ]]
P_rect_02 (3x4) =
 [[721.5377     0.       609.5593    44.85728 ]
 [  0.       721.5377   172.854      0.216379]
 [  0.         0.         1.         0.002746]]

f = 721.54 px, (cx, cy) = (609.56, 172.85) px, tx/f = +0.0622 m (cam2 가 cam0 에서 옆으로 떨어진 기준선)
pykitti b_rgb (cam2–cam3 기준선) = 0.5327 m  (KITTI 공식 약 0.54 m)

'T_cam0_velo_unrect' 가 우리 T_velo_to_cam 과 같은가 ? True  ← 이름만 다르다 (cam0 ← velo)


In [10]:
# 검증: 정답지 KittiCalibration 의 R_rect_00 (4x4), P_rect_02, image_size 와 비교
print("R_rect_00 allclose(atol=1e-6) ?", np.allclose(R_rect_00, calib.R_rect_00, atol=1e-6), f"| max |차이| = {np.abs(R_rect_00 - calib.R_rect_00).max():.1e} (정답지는 직교 보정)")
print("P_rect_02 array_equal ?", np.array_equal(P_rect_02, calib.P_rect_02), "(P 는 보정 대상이 아니라 완전히 같음)")
print("image_size (W, H):", calib.image_size, "== S_rect_02", tuple(int(v) for v in cc['S_rect_02']), "?", calib.image_size == tuple(int(v) for v in cc['S_rect_02']))
print("정답지 image_shape (H, W) =", calib.image_shape, " ← numpy 순서로 뒤집어 준다")

R_rect_00 allclose(atol=1e-6) ? True | max |차이| = 3.9e-08 (정답지는 직교 보정)
P_rect_02 array_equal ? True (P 는 보정 대상이 아니라 완전히 같음)
image_size (W, H): (1242, 375) == S_rect_02 (1242, 375) ? True
정답지 image_shape (H, W) = (375, 1242)  ← numpy 순서로 뒤집어 준다


### 조각 4. 세 행렬을 곱해 `P_velo_to_img` (3×4) 만들기 — readme 식과 `run_demoVelodyne.m`

- **검색어**: `kitti velodyne to image projection P_rect R_rect_00 velo_to_cam order`
- **출처**:
  - `readme.txt` "example transformations" 절 (206~231행) — KITTI(연구용) — 2026-09-08 — `notebooks_2nd/sources/kitti_devkit/devkit/readme.txt`
  - devkit `matlab/run_demoVelodyne.m` 24~31행 — KITTI(연구용) — 2026-09-08 — `notebooks_2nd/sources/kitti_devkit/devkit/matlab/run_demoVelodyne.m`
- **왜 이걸 가져오나**: 곱하는 **순서**와 "image_02 에 투영할 때도 `R_rect_00`(카메라 0 의 정류 회전) 을 쓴다" 는 규칙은 readme 와 데모 코드에만 있다. 잘못 곱하면 점이 엉뚱한 곳에 찍힌다.
- **읽을 때 볼 곳**: readme 217행 `Y = P_rect_xx * R_rect_00 * (R|T)_velo_to_cam * X` — 행렬 곱은 오른쪽부터 적용되므로 velo→cam0, 정류, 투영 순. MATLAB 29~31행: `R_cam_to_rect(1:3,1:3) = calib.R_rect{1}` — MATLAB 은 1-based 라 `{1}` = 카메라 0, `calib.P_rect{cam+1}` = 카메라 `cam` 의 `P_rect`.

readme.txt 원문 (213~231행):

```text
In order to transform a homogeneous point X = [x y z 1]' from the velodyne
coordinate system to a homogeneous point Y = [u v 1]' on image plane of
camera xx, the following transformation has to be applied:

Y = P_rect_xx * R_rect_00 * (R|T)_velo_to_cam * X

To transform a point X from GPS/IMU coordinates to the image plane:

Y = P_rect_xx * R_rect_00 * (R|T)_velo_to_cam * (R|T)_imu_to_velo * X

The matrices are:

- P_rect_xx (3x4):         rectfied cam 0 coordinates -> image plane
- R_rect_00 (4x4):         cam 0 coordinates -> rectified cam 0 coord.
- (R|T)_velo_to_cam (4x4): velodyne coordinates -> cam 0 coordinates
- (R|T)_imu_to_velo (4x4): imu coordinates -> velodyne coordinates

Note that the (4x4) matrices above are padded with zeros and:
R_rect_00(4,4) = (R|T)_velo_to_cam(4,4) = (R|T)_imu_to_velo(4,4) = 1.
```

run_demoVelodyne.m 원문 (21~31행):

```matlab
cam       = 2; % 0-based index
frame     = 0; % 0-based index

% load calibration
calib = loadCalibrationCamToCam(fullfile(calib_dir,'calib_cam_to_cam.txt'));
Tr_velo_to_cam = loadCalibrationRigid(fullfile(calib_dir,'calib_velo_to_cam.txt'));

% compute projection matrix velodyne->image plane
R_cam_to_rect = eye(4);
R_cam_to_rect(1:3,1:3) = calib.R_rect{1};
P_velo_to_img = calib.P_rect{cam+1}*R_cam_to_rect*Tr_velo_to_cam;
```

주의: readme 의 `Y = [u v 1]'` 는 실제로는 `[u·d, v·d, d]` 이고 세 번째 성분(깊이 d)으로 나눠야 픽셀 `(u, v)` 가 된다 (Phase 3 에서 사용).

In [11]:
# PORTED FROM kitti_devkit/devkit/matlab/run_demoVelodyne.m :: 28~31행 (MATLAB → numpy), KITTI
def velo_to_img_matrix(P_rect, R_rect_00_3x3, Tr_velo_to_cam):
    """P_velo_to_img = P_rect{cam+1} * R_cam_to_rect * Tr_velo_to_cam  (MATLAB 의 '*' 는 행렬 곱 → numpy '@')"""
    R_cam_to_rect = np.eye(4)                     # R_cam_to_rect = eye(4);
    R_cam_to_rect[0:3, 0:3] = R_rect_00_3x3       # R_cam_to_rect(1:3,1:3) = calib.R_rect{1};   ({1} = 카메라 0, 1-based)
    return P_rect @ R_cam_to_rect @ Tr_velo_to_cam

In [12]:
cam = 2                                                                    # 0-based index (MATLAB 데모와 같음)
P_velo_to_img = velo_to_img_matrix(cc[f'P_rect_{cam:02d}'].reshape(3, 4), cc['R_rect_00'].reshape(3, 3), T_velo_to_cam)
print("P_velo_to_img (3x4) =\n", P_velo_to_img)

# 같은 식을 조각 3 의 4x4 R_rect_00 으로 써도 동일 (readme 217행 그대로)
P_check = P_rect_02 @ R_rect_00 @ T_velo_to_cam
print("readme 식 P_rect_02 @ R_rect_00 @ T_velo_to_cam 과 동일 ?", np.array_equal(P_velo_to_img, P_check))

# 순서를 바꾸면? (흔한 실수: velo→cam 을 나중에 곱함) — 곱 자체는 성립하지 않거나(shape) 값이 완전히 다르다
try:
    wrong = P_rect_02 @ T_velo_to_cam @ R_rect_00
    print("순서를 바꾼 P (shape 은 같지만 값이 다름): max |차이| =", f"{np.abs(wrong - P_velo_to_img).max():.2f}")
except ValueError as e:
    print("순서를 바꾸면 ValueError:", e)

# pykitti 의 다른 길: K_cam2 @ T_cam2_velo[:3]  (기준선 tx/f 를 T2 로 옮겨 "cam2 rect 프레임" 을 만든 뒤 K 로 투영)
P_pykitti = data['K_cam2'] @ data['T_cam2_velo'][:3]
print("\npykitti 방식 K_cam2 @ T_cam2_velo 와의 차이 (원소별 max) =", f"{np.abs(P_pykitti - P_velo_to_img).max():.4f}",
      "\n  → P_rect_02 의 4번째 열 [tx, ty, tz] 중 pykitti 는 tx 만 T2 로 옮기고 ty, tz 는 버린다. 차이는 ty =", P_rect_02[1, 3].round(4),
      "→ 픽셀로는 v 가 ty/d 만큼 (10 m 에서 ~0.02 px) 차이. 실용상 무시되지만 '같은 식' 은 아니다.")

# 전방 10 m 점을 두 식으로 투영해 픽셀 차이 확인
X = np.array([10.0, 0.0, 0.0, 1.0])
for name, P in (("readme 식", P_velo_to_img), ("pykitti 식", P_pykitti)):
    y = P @ X
    print(f"  {name:10s} velo (10,0,0) → d = {y[2]:.3f} m, (u, v) = ({y[0]/y[2]:.3f}, {y[1]/y[2]:.3f}) px")

P_velo_to_img (3x4) =
 [[ 609.695409 -721.421597   -1.251259 -123.041806]
 [ 180.384202    7.644798 -719.651474 -101.016688]
 [   0.999945    0.000124    0.010451   -0.269387]]
readme 식 P_rect_02 @ R_rect_00 @ T_velo_to_cam 과 동일 ? True
순서를 바꾼 P (shape 은 같지만 값이 다름): max |차이| = 12.31

pykitti 방식 K_cam2 @ T_cam2_velo 와의 차이 (원소별 max) = 0.2164 
  → P_rect_02 의 4번째 열 [tx, ty, tz] 중 pykitti 는 tx 만 T2 로 옮기고 ty, tz 는 버린다. 차이는 ty = 0.2164 → 픽셀로는 v 가 ty/d 만큼 (10 m 에서 ~0.02 px) 차이. 실용상 무시되지만 '같은 식' 은 아니다.
  readme 식   velo (10,0,0) → d = 9.730 m, (u, v) = (613.964, 175.007) px
  pykitti 식  velo (10,0,0) → d = 9.727 m, (u, v) = (614.137, 175.034) px


In [13]:
# 검증: 정답지 calib.P_velo_to_img 와 비교. 정답지는 R, R_rect_00 을 직교 보정하므로 ~1e-5 수준 차이가 남는다.
diff = np.abs(P_velo_to_img - calib.P_velo_to_img)
print("allclose(rtol=1e-6, atol=1e-4) ?", np.allclose(P_velo_to_img, calib.P_velo_to_img, rtol=1e-6, atol=1e-4),
      f"| max |차이| = {diff.max():.1e} (원소 크기 ~{np.abs(calib.P_velo_to_img).max():.0f} 대비 상대 {diff.max()/np.abs(calib.P_velo_to_img).max():.1e})")

# 픽셀로 환산하면 얼마나 다른가: 임의의 전방 점 1만 개
rng = np.random.default_rng(0)
pts = rng.uniform([2, -30, -3], [80, 30, 3], size=(10_000, 3))
pts_h = np.hstack([pts, np.ones((len(pts), 1))])
y_mine, y_ref = pts_h @ P_velo_to_img.T, pts_h @ calib.P_velo_to_img.T
uv_mine, uv_ref = y_mine[:, :2] / y_mine[:, 2:], y_ref[:, :2] / y_ref[:, 2:]
print(f"픽셀 (u, v) 최대 차이 = {np.abs(uv_mine - uv_ref).max():.1e} px  ← 투영에는 영향 없음")

allclose(rtol=1e-6, atol=1e-4) ? True | max |차이| = 1.6e-05 (원소 크기 ~721 대비 상대 2.3e-08)
픽셀 (u, v) 최대 차이 = 4.6e-04 px  ← 투영에는 영향 없음


### 조각 5. 역변환 `T⁻¹ = [[Rᵀ, −Rᵀt], [0, 1]]` 과 텍스트 반올림 함정 (SVD 직교 보정)

- **검색어**: `inverse rigid transformation matrix rotation transpose`, `nearest orthogonal matrix svd`
- **출처**:
  - Wikipedia "Orthogonal matrix" (`Qᵀ = Q⁻¹`) — https://en.wikipedia.org/wiki/Orthogonal_matrix — CC BY-SA 4.0 — 2026-09-08 — `notebooks_2nd/sources/wikipedia/orthogonal_matrix.txt`
  - Wikipedia "Affine transformation — Augmented matrix" (`[y;1] = [[A,b],[0,1]][x;1]` ⇔ `y = Ax + b`) — `notebooks_2nd/sources/wikipedia/affine_transformation.txt`
  - Wikipedia "Orthogonal Procrustes problem — Solution" (`M = UΣVᵀ → R = UVᵀ`) — https://en.wikipedia.org/wiki/Orthogonal_Procrustes_problem — CC BY-SA 4.0 — 2026-09-08 — `notebooks_2nd/sources/wikipedia/orthogonal_procrustes.txt`
  - pykitti `raw.py` 214~218행: 실무 코드는 그냥 `np.linalg.inv(data['T_cam0_velo'])` 를 쓴다 (조각 3 의 A 셀에 원문 포함) — MIT
- **왜 이걸 가져오나**: Phase 5·6 에서 카메라(rect) 프레임의 결과를 Velodyne 으로 되돌린다. `np.linalg.inv` 도 되지만, 강체 변환은 `Rᵀ` 로 역을 쓸 수 있고 의미("회전 되돌리고 이동 빼기")가 드러난다. 단, **파일의 R 이 정확히 직교가 아니면** `Rᵀ` 역은 정확하지 않다.
- **읽을 때 볼 곳**: `y = Ax + b` 의 역은 `x = A⁻¹(y − b) = A⁻¹y − A⁻¹b` → `[[A⁻¹, −A⁻¹b],[0,1]]`. 회전이면 `A⁻¹ = Aᵀ`. Procrustes: 가장 가까운 직교 행렬은 SVD 의 `U Vᵀ`.

유도 (Wikipedia 두 문장을 합친 것):

$$T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix},\quad p' = Rp + t \;\Rightarrow\; p = R^{-1}(p' - t) = R^T p' - R^T t \;\Rightarrow\; T^{-1} = \begin{bmatrix} R^T & -R^T t \\ 0 & 1 \end{bmatrix}$$

In [14]:
# PORTED FROM Wikipedia "Orthogonal matrix" (Qᵀ = Q⁻¹) + "Affine transformation" (augmented matrix), CC BY-SA 4.0
def inverse_rigid(T):
    """T = [[R, t], [0, 1]]  →  T⁻¹ = [[Rᵀ, −Rᵀ t], [0, 1]]   (R 이 직교라는 전제)"""
    R, t = T[:3, :3], T[:3, 3]
    Ti = np.eye(4)
    Ti[:3, :3] = R.T
    Ti[:3, 3] = -R.T @ t
    return Ti

# PORTED FROM Wikipedia "Orthogonal Procrustes problem" :: Solution (M = U Σ Vᵀ → R = U Vᵀ), CC BY-SA 4.0
def nearest_orthogonal(M):
    """M 에 가장 가까운 직교 행렬 (Frobenius 노름 기준)."""
    U, _, Vt = np.linalg.svd(M)
    return U @ Vt

In [15]:
T_velo_to_rect = R_rect_00 @ T_velo_to_cam                  # velo → rect (4x4). 조각 3·2 의 행렬 그대로 (보정 없음)
p = np.array([70.0, 40.0, 3.0, 1.0])                         # 먼 점 (70 m) — 오차가 커지는 곳

for name, inv in (("np.linalg.inv  (pykitti 방식)", np.linalg.inv), ("Rᵀ 공식      (파일 R 그대로)", inverse_rigid)):
    back = inv(T_velo_to_rect) @ (T_velo_to_rect @ p)
    print(f"{name:32s} round-trip 오차 = {np.abs(back - p).max():.2e} m")

R, Rr = T_velo_to_cam[:3, :3], R_rect_00[:3, :3]
print(f"\n원인: 파일의 회전이 정확히 직교가 아님. |R Rᵀ − I|max = {np.abs(R @ R.T - np.eye(3)).max():.1e}, |R_rect Rᵀ_rect − I|max = {np.abs(Rr @ Rr.T - np.eye(3)).max():.1e} (유효숫자 7자리 반올림)")

# 보정: 가장 가까운 직교 행렬로 바꾼 뒤 다시
R_fix, Rr_fix = nearest_orthogonal(R), nearest_orthogonal(Rr)
print(f"보정 후 |R Rᵀ − I|max = {np.abs(R_fix @ R_fix.T - np.eye(3)).max():.1e}, |R_fix − R|max = {np.abs(R_fix - R).max():.1e}, det = {np.linalg.det(R_fix):+.6f} (반사가 아닌 회전인지 확인)")
T_velo_to_rect_fix = transform_from_rot_trans(Rr_fix, np.zeros(3)) @ transform_from_rot_trans(R_fix, vc['T'])
back = inverse_rigid(T_velo_to_rect_fix) @ (T_velo_to_rect_fix @ p)
print(f"보정 후 Rᵀ 공식 round-trip 오차 = {np.abs(back - p).max():.2e} m  ← 허용치 1e-6 m 안으로")

np.linalg.inv  (pykitti 방식)      round-trip 오차 = 2.84e-14 m
Rᵀ 공식      (파일 R 그대로)            round-trip 오차 = 1.85e-06 m

원인: 파일의 회전이 정확히 직교가 아님. |R Rᵀ − I|max = 9.0e-08, |R_rect Rᵀ_rect − I|max = 7.9e-08 (유효숫자 7자리 반올림)
보정 후 |R Rᵀ − I|max = 8.9e-16, |R_fix − R|max = 4.5e-08, det = +1.000000 (반사가 아닌 회전인지 확인)
보정 후 Rᵀ 공식 round-trip 오차 = 7.11e-14 m  ← 허용치 1e-6 m 안으로


In [16]:
# 검증: 정답지 invert_rigid / nearest_rotation 과 비교, 정답지 round-trip
from perceptrack3d.geometry.transforms import invert_rigid, nearest_rotation

print("inverse_rigid == 정답지 invert_rigid ?", np.allclose(inverse_rigid(T_velo_to_rect_fix), invert_rigid(T_velo_to_rect_fix), atol=1e-12))
print("nearest_orthogonal == 정답지 nearest_rotation ?", np.allclose(nearest_orthogonal(R), nearest_rotation(R), atol=1e-12),
      " (차이: 정답지는 det < 0 이면 마지막 열 부호를 뒤집어 반사를 회전으로 만든다. 이 파일은 det > 0 이라 결과 동일)")
print("보정한 T_velo_to_rect == 정답지 calib.T_velo_to_rect ?", np.allclose(T_velo_to_rect_fix, calib.T_velo_to_rect, atol=1e-12))

rng = np.random.default_rng(0)
pts = rng.uniform([-80, -40, -3], [80, 40, 3], size=(100_000, 3))
back = calib.rect_to_velo(calib.velo_to_rect(pts))
print(f"정답지 round-trip (10만 점, |x| ≤ 80 m) max 오차 = {np.abs(back - pts).max():.2e} m")
print(f"|invert_rigid(T) @ T − I|max = {np.abs(invert_rigid(calib.T_velo_to_rect) @ calib.T_velo_to_rect - np.eye(4)).max():.1e}")

inverse_rigid == 정답지 invert_rigid ? True
nearest_orthogonal == 정답지 nearest_rotation ? True  (차이: 정답지는 det < 0 이면 마지막 열 부호를 뒤집어 반사를 회전으로 만든다. 이 파일은 det > 0 이라 결과 동일)
보정한 T_velo_to_rect == 정답지 calib.T_velo_to_rect ? True
정답지 round-trip (10만 점, |x| ≤ 80 m) max 오차 = 8.53e-14 m
|invert_rigid(T) @ T − I|max = 1.1e-15


### 조각 6. 부호 확인 실험 — 아는 점 몇 개로 축 방향 검사

- **검색어**: `kitti velodyne camera coordinate system axes x forward y left z up`
- **출처**: `readme.txt` "Coordinate Systems" 절 (149~160행) — KITTI(연구용) — 2026-09-08 — `notebooks_2nd/sources/kitti_devkit/devkit/readme.txt`; 센서 배치 그림 https://www.cvlibs.net/datasets/kitti/setup.php
- **왜 이걸 가져오나**: 행렬을 다 붙인 뒤 "맞는지" 를 확인하는 가장 싼 방법은 답을 아는 점을 넣어 보는 것이다. 축 정의는 readme 에만 있다.
- **읽을 때 볼 곳**: `Camera: x: right, y: down, z: forward` / `Velodyne: x: forward, y: left, z: up` → velo +x → cam +z, velo +y(좌) → cam −x, velo +z(상) → cam −y. 그리고 조각 2 에서 본 T: Velodyne 원점은 카메라 0.27 m 뒤 → velo 원점의 rect z ≈ −0.27.

In [17]:
# ORIGINAL: kitti_devkit/devkit/readme.txt :: Coordinate Systems (149~160행), KITTI
COORDINATE_SYSTEMS = '''
Coordinate Systems
==================

The coordinate systems are defined the following way, where directions
are informally given from the drivers view, when looking forward onto
the road:

  - Camera:   x: right,   y: down,  z: forward
  - Velodyne: x: forward, y: left,  z: up
  - GPS/IMU:  x: forward, y: left,  z: up

All coordinate systems are right-handed.
'''
print(COORDINATE_SYSTEMS)


Coordinate Systems

The coordinate systems are defined the following way, where directions
are informally given from the drivers view, when looking forward onto
the road:

  - Camera:   x: right,   y: down,  z: forward
  - Velodyne: x: forward, y: left,  z: up
  - GPS/IMU:  x: forward, y: left,  z: up

All coordinate systems are right-handed.



In [18]:
def apply_T(T, pts):                                   # (N,3) 점에 4x4 강체 변환 적용 → (N,3)
    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    return (pts_h @ T.T)[:, :3]

T_velo_to_rect = T_velo_to_rect_fix                    # 조각 5 에서 보정한 것
test = np.array([[0.0, 0.0, 0.0], [10.0, 0.0, 0.0], [10.0, 5.0, 0.0], [10.0, 0.0, 3.0]])
labels = ["velo 원점", "전방 10 m", "전방 10, 좌 5 m", "전방 10, 위 3 m"]
expect = ["rect z ≈ −0.27 (카메라 뒤)", "rect z ≈ 9.73, x ≈ y ≈ 0", "rect x ≈ −5 (좌 = 카메라 −x)", "rect y ≈ −3 (상 = 카메라 −y)"]

rect = apply_T(T_velo_to_rect, test)
print(f"{'velo 점':18s} {'rect (x, y, z)':>28s}   기대")
for lab, r, e in zip(labels, rect, expect):
    print(f"{lab:18s} {str(r.round(3)):>28s}   {e}")

checks = {
    "velo 원점 → rect z < 0 (카메라 뒤)": rect[0, 2] < 0 and abs(rect[0, 2] + 0.27) < 0.02,
    "전방 10 m → rect z ≈ 9.73":         abs(rect[1, 2] - 9.73) < 0.02,
    "velo +y(좌) → rect x 음수":          rect[2, 0] < -4.9,
    "velo +z(상) → rect y 음수":          rect[3, 1] < -2.9,
}
for k, v in checks.items():
    print(f"  [{'OK' if v else 'FAIL'}] {k}")
assert all(checks.values()), "축 부호가 readme 정의와 맞지 않음"

cam0_in_velo = inverse_rigid(T_velo_to_cam)[:3, 3]
print("\ncam0 원점 (velo 프레임) =", cam0_in_velo.round(3), "m → 카메라는 LiDAR 보다 앞(+x), 아래(−z)")

velo 점                           rect (x, y, z)   기대
velo 원점                  [-0.003 -0.075 -0.272]   rect z ≈ −0.27 (카메라 뒤)
전방 10 m                  [-0.     0.029  9.727]   rect z ≈ 9.73, x ≈ y ≈ 0
전방 10, 좌 5 m             [-5.     0.082  9.728]   rect x ≈ −5 (좌 = 카메라 −x)
전방 10, 위 3 m             [-0.032 -2.97   9.759]   rect y ≈ −3 (상 = 카메라 −y)
  [OK] velo 원점 → rect z < 0 (카메라 뒤)
  [OK] 전방 10 m → rect z ≈ 9.73
  [OK] velo +y(좌) → rect x 음수
  [OK] velo +z(상) → rect y 음수

cam0 원점 (velo 프레임) = [ 0.273 -0.002 -0.072] m → 카메라는 LiDAR 보다 앞(+x), 아래(−z)


In [19]:
# 검증: 정답지 calib.velo_to_rect 와 비교 (atol 1e-6)
ref = calib.velo_to_rect(test)
print("np.allclose(atol=1e-6) ?", np.allclose(rect, ref, atol=1e-6), f"| max |차이| = {np.abs(rect - ref).max():.1e} m")
print("정답지 rect:\n", ref.round(6))

np.allclose(atol=1e-6) ? True | max |차이| = 0.0e+00 m
정답지 rect:
 [[-0.002797 -0.075109 -0.272133]
 [-0.000449  0.029385  9.727321]
 [-5.00017   0.082212  9.727943]
 [-0.03214  -2.970283  9.758675]]


### 이어붙이기 — `load_calib(calib_dir)` 한 함수로

조각 1(파싱) → 5(직교 보정) → 2(4×4) → 3(R_rect 4×4, P_rect_02) → 4(곱) 을 이어 정답지 `KittiCalibration.from_dir` 과 같은 것을 만든다.

In [20]:
def load_calib(calib_dir):
    """calib_dir 의 두 텍스트 → dict(T_velo_to_cam 4x4, R_rect_00 4x4, P_rect_02 3x4, P_velo_to_img 3x4, image_size (W,H))."""
    calib_dir = Path(calib_dir)
    vc = read_calib_file(calib_dir / 'calib_velo_to_cam.txt')                        # 조각 1
    cc = read_calib_file(calib_dir / 'calib_cam_to_cam.txt')
    R = nearest_orthogonal(vc['R'].reshape(3, 3))                                     # 조각 5: 텍스트 반올림 보정
    R_rect = nearest_orthogonal(cc['R_rect_00'].reshape(3, 3))
    T_velo_to_cam = transform_from_rot_trans(R, vc['T'])                              # 조각 2
    R_rect_00 = np.eye(4); R_rect_00[0:3, 0:3] = R_rect                               # 조각 3 (readme 230~231행)
    P_rect_02 = np.reshape(cc['P_rect_02'], (3, 4))                                   # 조각 3 (파일 키 이름 그대로)
    return {
        "T_velo_to_cam": T_velo_to_cam,
        "R_rect_00": R_rect_00,
        "P_rect_02": P_rect_02,
        "P_velo_to_img": P_rect_02 @ R_rect_00 @ T_velo_to_cam,                        # 조각 4 (readme 217행)
        "image_size": tuple(int(v) for v in cc['S_rect_02']),
    }

mine = load_calib(CALIB_DIR)
ref = KittiCalibration.from_dir(CALIB_DIR)
for k, r in (("T_velo_to_cam", ref.T_velo_to_cam), ("R_rect_00", ref.R_rect_00), ("P_rect_02", ref.P_rect_02), ("P_velo_to_img", ref.P_velo_to_img)):
    print(f"{k:14s} shape {mine[k].shape} | 정답지와 allclose(atol=1e-9) ? {np.allclose(mine[k], r, atol=1e-9)} | max |차이| = {np.abs(mine[k] - r).max():.1e}")
print(f"{'image_size':14s} {mine['image_size']} == {ref.image_size} ? {mine['image_size'] == ref.image_size}")

T_velo_to_cam  shape (4, 4) | 정답지와 allclose(atol=1e-9) ? True | max |차이| = 0.0e+00
R_rect_00      shape (4, 4) | 정답지와 allclose(atol=1e-9) ? True | max |차이| = 0.0e+00
P_rect_02      shape (3, 4) | 정답지와 allclose(atol=1e-9) ? True | max |차이| = 0.0e+00
P_velo_to_img  shape (3, 4) | 정답지와 allclose(atol=1e-9) ? True | max |차이| = 1.1e-13
image_size     (1242, 375) == (1242, 375) ? True


### 여기서 배우는 것

- **이름의 방향**: 파일 키 `P_rect_02` ↔ pykitti `P_rect_20`, pykitti `T_cam0_velo`(velo → cam0) ↔ 정답지 `T_velo_to_cam`. 같은 행렬을 라이브러리마다 반대로 이름 짓는다. 붙이기 전에 "무엇에서 무엇으로" 를 한 점으로 확인한다 (조각 6).
- **곱 순서와 R_rect_00**: `P_rect_02 @ R_rect_00 @ T_velo_to_cam` 은 오른쪽부터 적용된다. image_02 에 투영해도 `R_rect_02` 가 아니라 **`R_rect_00`** 이다 (readme 217행, MATLAB `R_rect{1}`).
- **같은 결과 ≠ 같은 식**: pykitti 는 기준선 tx 만 `T_cam2_velo` 로 옮기고 `P_rect_02` 의 ty, tz 는 버린다. 픽셀 차이 ~0.02 px 라 실무에선 무시되지만, 검증 없이 "같다" 고 믿으면 안 된다.
- **텍스트 반올림**: 유효숫자 7자리 회전은 `R Rᵀ − I ≈ 8e-8`. `np.linalg.inv` 는 괜찮지만 `Rᵀ` 역은 70 m 에서 1.8e-6 m 오차. SVD `U Vᵀ` 보정으로 1e-14 로 내려간다. 보정 자체는 값 변화 1e-7, 투영 영향 1e-5 px.
- **shape 순서**: `S_rect_02` 는 (W, H) = (1242, 375), numpy 이미지는 (H, W). 4×4 확장은 `eye(4)` 에 넣어 `[3, 3] = 1` 을 유지.

### 자가 점검 3문제

1. `P_rect_02 @ R_rect_00 @ T_velo_to_cam @ x` 에서 Velodyne 점 `x` 에 **가장 먼저** 적용되는 행렬은 무엇이고, 그 결과는 어느 프레임의 점인가?
2. pykitti 의 `data['T_cam2_velo']` 를 그대로 가져와 `K_cam2 @ T_cam2_velo[:3]` 로 투영하면 readme 식과 무엇이 달라지는가? 픽셀로는 대략 얼마인가?
3. 파일의 `R` 을 그대로 `Rᵀ` 로 역변환하면 70 m 점의 round-trip 오차가 왜 1e-6 m 대로 커지는가? 무엇으로 고치며, 고친 R 은 원래와 얼마나 다른가?

<details>
<summary>답</summary>

1. `T_velo_to_cam` (행렬 곱은 오른쪽부터). 결과는 카메라 0(cam0, x 우·y 아래·z 전방) 프레임의 점. 그다음 `R_rect_00` 으로 정류 프레임, 마지막 `P_rect_02` 로 이미지 동차 좌표 `[u·d, v·d, d]`.
2. `P_rect_02` 의 4번째 열 중 `tx` 만 `T2` 로 옮기고 `ty`(0.216), `tz`(0.0027) 를 버린다. 그래서 `v` 가 `ty/d` 만큼 (10 m 에서 약 0.02 px) 달라진다.
3. 텍스트 파일의 R 은 유효숫자 7자리라 `R Rᵀ ≠ I` (오차 ~8e-8). `Rᵀ` 를 역으로 쓰면 그 오차가 거리(70 m)에 비례해 ~1.8e-6 m 가 된다. SVD 로 가장 가까운 직교 행렬 `U Vᵀ` 로 바꾸면 (orthogonal Procrustes) 오차가 1e-14 m 로 떨어지고, R 의 값 변화는 ~1e-7 뿐이다.

</details>